---
## 1. Імпорт бібліотек

У цьому розділі підключаються всі необхідні бібліотеки:

- **`chardet`** — автоматичне визначення кодування CSV-файлу (важливо для файлів з кириличними символами)
- **`pandas` / `numpy`** — основні інструменти обробки табличних даних та числових обчислень
- **`matplotlib` / `seaborn`** — візуалізація результатів
- **`scipy.sparse`** — робота з розрідженими матрицями (TF-IDF породжує матриці з мільйонами нульових елементів)
- **`sklearn`** — весь стек машинного навчання: векторизація тексту, масштабування ознак, класифікатори, метрики

Константи `RANDOM_STATE` та `N_QUINTILES` фіксують відтворюваність і кількість цільових класів.

In [ ]:
import chardet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, label_binarize, OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

RANDOM_STATE = 42
N_QUINTILES  = 5
print('Усі бібліотеки імпортовано успішно')

---
## 2. Завантаження даних

Файл зчитується у два кроки:

1. **Визначення кодування** (`chardet.detect`): ми читаємо перші 100 КБ файлу у бінарному режимі і автоматично визначаємо кодування (наприклад, `windows-1251` або `utf-8`). Це критично для україномовних текстів, де неправильне кодування призводить до нечитабельних символів.

2. **Зчитування CSV** з роздільником `;` та автоматичним розпізнаванням дат у колонці `receivedDateTime`. Pandas перетворює рядки з датами на об'єкти `datetime64`, що дозволяє надалі легко витягувати годину, день тижня тощо.

Після завантаження виводимо форму датасету та перші рядки для первинного огляду.

In [ ]:
with open('./data/appeals_2026-04-02.csv', 'rb') as f:
    enc = chardet.detect(f.read(100000))
print(enc)
        
df_raw = pd.read_csv(
    './data/appeals_2026-04-02.csv',
    encoding=enc['encoding'],
    sep=';',
    parse_dates=['receivedDateTime']
)
print(f'Форма датасету: {df_raw.shape}')
df_raw.head(3)

In [ ]:
df_raw["result"].unique()

In [ ]:
df_raw.describe()

In [ ]:
df_raw.dtypes

---
## 3. Конструювання цільової змінної — Результат розгляду

**Цільова змінна**: `result` — фактичний результат розгляду звернення.

### Обробка NaN
Значення `NaN` у полі `result` означає, що звернення **ще не розглянуто**.  
Такі записи **виключаються** з датасету — ми прогнозуємо лише для вже розглянутих звернень.

### Кодування міток
Унікальні значення `result` кодуються у числові мітки через `LabelEncoder`.  
Розподіл класів відображається на гістограмі нижче.

In [ ]:
# Фільтрація: залишаємо лише розглянуті звернення (result не NaN)
df_resolved = df_raw.dropna(subset=['result']).copy()
print(f'Розглянуті: {len(df_resolved):,} / {len(df_raw):,}')
print(f'Нерозглянуті (NaN result) виключено: {len(df_raw) - len(df_resolved):,}')

# Унікальні значення результату
print('\nУнікальні значення result:')
print(df_resolved['result'].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
counts = df_resolved['result'].value_counts()
bars = ax.bar(range(len(counts)), counts.values,
              color=['#27ae60','#2980b9','#f39c12','#e67e22','#e74c3c'][:len(counts)],
              alpha=0.88, edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5, str(v), ha='center', fontsize=9)
ax.set_title('Цільова змінна: Розподіл результатів розгляду')
ax.set_xlabel('Результат')
ax.set_ylabel('Кількість звернень')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Конструювання ознак

Ознаки розділені на **два повністю ізольованих набори**:

| Стратегія | Ознаки | Використовується в |
|---|---|---|
| **Тільки текст** | TF-IDF по `type + kind + content` | NB, LR |
| **Тільки таблиця** | Часові + адресні + організаційні — *нуль TF-IDF* | RF, HistGB |

### Часові ознаки
`hour`, `dayofweek`, `month`, `is_weekend`, `is_night`, `is_business_hours`,  
синусо-косинусне кодування годин і дня тижня.  
**Примітка**: `weekofyear` **виключено** — він є монотонно зростаючим і корелює з часовим розбиттям.

### Адресні та організаційні ознаки
`postcode_known`, `street_len`, `has_building`, `org_id_str`, `content_len`.

###  ВИПРАВЛЕННЯ 4: Frequency Encoding для `org_id_str`
Замість OHE (що породжує тисячі колонок і заучує конкретні id) використовується  
**Frequency Encoding** — кожна організація замінюється на кількість її звернень у train.  
Це зберігає інформацію про «великі» та «маленькі» організації без перенавчання на id.

In [ ]:
df = df_resolved.copy()

# Часові ознаки (weekofyear ВИКЛЮЧЕНО — є монотонним проксі часового розбиття)
df['hour']              = df['receivedDateTime'].dt.hour
df['dayofweek']         = df['receivedDateTime'].dt.dayofweek
df['month']             = df['receivedDateTime'].dt.month
df['is_weekend']        = (df['dayofweek'] >= 5).astype(int)
df['is_night']          = ((df['hour'] < 7) | (df['hour'] >= 22)).astype(int)
df['is_business_hours'] = ((df['hour'] >= 9) & (df['hour'] <= 17)
                            & (df['dayofweek'] < 5)).astype(int)
df['hour_sin']          = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']          = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']           = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']           = np.cos(2 * np.pi * df['dayofweek'] / 7)

# Адресні ознаки
df['postcode_known']  = df['addressPostCode'].notna().astype(int)
df['postcode']        = df['addressPostCode'].fillna(0).astype(int).astype(str)
df['street_len']      = df['addressThoroughfare'].fillna('').str.len()
df['has_building']    = df['addressLocatorDesignator'].notna().astype(int)

# Ознаки організації та вмісту
df['org_id_str']   = df['organizationId'].fillna(0).astype(int).astype(str)
df['content_len']  = df['content'].fillna('').str.len()

# Текст для TF-IDF моделей
df['input_text'] = (df['type'].fillna('') + ' | ' +
                    df['kind'].fillna('') + ' | ' +
                    df['content'].fillna(''))

# Цільова змінна: result (рядкові мітки -> числові)
TARGET = 'result'
le = LabelEncoder()
df['label'] = le.fit_transform(df[TARGET])
classes = le.classes_
class_names = list(classes)

print(f'Класи ({len(classes)}): {list(classes)}')
print(f'\nРозподіл міток:')
print(df['label'].value_counts().rename(index=dict(enumerate(classes))))
print(df[['hour','dayofweek','is_weekend','is_night','postcode_known',
          'street_len','content_len']].describe().round(2))

---
## 5. Часовий поділ на навчальну і тестову вибірки

**Часовий розподіл**: навчання на січні–лютому 2026, тест на березні 2026.

**Важливо**: агрегати за організацією обчислюються лише на навчальній вибірці  
(щоб уникнути витоку через `result`). Використовуємо **структурні агрегати**:
кількість звернень організації (`org_volume`) та частоту звернень за категорією (`org_kind_count`).

**Frequency Encoding** для `org_id_str` також обчислюється тут — виключно на train.

In [ ]:
train_df = df[df['month'] <= 2].copy()
test_df  = df[df['month'] == 3].copy()

print(f'Навчальна вибірка (Січень-Лютий): {len(train_df):,} записів')
print(f'Тестова вибірка  (Березень)     : {len(test_df):,} записів')

assert train_df['receivedDateTime'].max() < test_df['receivedDateTime'].min(), \
    'Виявлено часовий витік!'
print('Часового витоку не виявлено ')

# Агрегати виключно по навчальним даним (без використання цільової змінної — без витоку)
org_volume = (
    train_df
    .groupby('org_id_str')
    .size()
    .reset_index(name='org_volume')
)

org_kind_counts = (
    train_df
    .groupby(['org_id_str', 'kind'])
    .size()
    .reset_index(name='org_kind_count')
)

#  ВИПРАВЛЕННЯ 4: Frequency Encoding для org_id_str (лише на train)
# Замість OHE, яке створює тисячі колонок і заучує конкретні id,
# замінюємо кожну організацію на кількість її звернень у train.
# Нові організації у тесті отримають значення 0 (замість вектора нулів у OHE).
org_freq_map = train_df['org_id_str'].value_counts().to_dict()

def add_agg_features(dframe):
    d = dframe.copy()
    d = d.merge(org_volume, on='org_id_str', how='left')
    d = d.merge(org_kind_counts, on=['org_id_str', 'kind'], how='left')
    d['org_volume'].fillna(0, inplace=True)
    d['org_kind_count'].fillna(0, inplace=True)
    # Frequency encoding: org_freq = кількість звернень цієї org у train
    d['org_freq'] = d['org_id_str'].map(org_freq_map).fillna(0)
    return d

train_df = add_agg_features(train_df)
test_df  = add_agg_features(test_df)

print(f'\nОрганізаційний обсяг (топ за кількістю звернень):')
print(org_volume.sort_values('org_volume', ascending=False).head(6))
print(f'\nFrequency Encoding — нових org у тесті (отримають 0): '
      f"{(test_df['org_freq'] == 0).sum()}")

---
## 6. Побудова матриць ознак

П'ять матриць для різних моделей:

1. **TF-IDF** — символьний n-gram (2–4) по конкатенованому тексту, `min_df=5` (виправлення #5)
2. **Таблична розріджена** — числові (scaled) + OHE лише для `type`, `kind`, `postcode`  
   (без `org_id_str` — він тепер у `org_freq` як числова ознака)
3. **Таблична щільна** — для HistGradientBoosting
4. **Злита розріджена** — TF-IDF + таблична (для LogReg)
5. **Злита щільна (SVD)** — SVD(100) тексту + таблична **← нова модель!**

###  ВИПРАВЛЕННЯ 3: `X_train_fused_dense` тепер використовується в моделі
###  ВИПРАВЛЕННЯ 5: `min_df=5` замість `min_df=1`

In [ ]:
#  ВИПРАВЛЕННЯ 5: min_df=5 — видаляємо токени, що зустрічаються менше 5 разів
# (у старій версії min_df=1 давав шум від унікальних імен та помилок друку)
tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 4),
    max_features=15000, sublinear_tf=True,
    min_df=5  # ← виправлено з 1 (за замовчуванням)
)
X_train_tfidf = tfidf.fit_transform(train_df['input_text'])
X_test_tfidf  = tfidf.transform(test_df['input_text'])

y_train = train_df['label'].values
y_test  = test_df['label'].values

print(f'TF-IDF: навчальна {X_train_tfidf.shape}, тестова {X_test_tfidf.shape}')

#  ВИПРАВЛЕННЯ 4: org_id_str виключено з CAT_COLS — тепер у org_freq як числова
NUM_COLS = [
    'hour', 'dayofweek', 'is_weekend', 'is_night', 'is_business_hours',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'postcode_known', 'street_len', 'has_building', 'content_len',
    'org_volume', 'org_kind_count',
    'org_freq'  # ← нова ознака: Frequency Encoding замість OHE org_id_str
]
# org_id_str прибрано з CAT_COLS
CAT_COLS = ['type', 'kind', 'postcode']

for col in CAT_COLS:
    train_df[col] = train_df[col].fillna('UNKNOWN')
    test_df[col]  = test_df[col].fillna('UNKNOWN')

# Категоріальне кодування (тільки type, kind, postcode)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(train_df[CAT_COLS])
X_test_cat  = ohe.transform(test_df[CAT_COLS])

# Масштабування числових ознак
scaler = StandardScaler(with_mean=False)
X_train_num = csr_matrix(scaler.fit_transform(
    train_df[NUM_COLS].fillna(0).values))
X_test_num  = csr_matrix(scaler.transform(
    test_df[NUM_COLS].fillna(0).values))

# 3. Таблична матриця (розріджена і щільна)
X_train_tab = hstack([X_train_num, X_train_cat])
X_test_tab  = hstack([X_test_num,  X_test_cat])

X_train_tab_dense = np.hstack([
    train_df[NUM_COLS].fillna(np.nan).values,
    X_train_cat.toarray()
])
X_test_tab_dense = np.hstack([
    test_df[NUM_COLS].fillna(np.nan).values,
    X_test_cat.toarray()
])

# 4. Злита матриця (TF-IDF + таблична — розріджена, для LogReg)
#  ВИПРАВЛЕННЯ 6: Масштабування TF-IDF у злитій матриці
# Проблема: TF-IDF має 3616 колонок vs 75 табличних → займає 98% L2-норми,
# що змушує LogReg ігнорувати табличні ознаки (96.4% ваг іде в текст).
# Рішення: множимо TF-IDF на коефіцієнт alpha = sqrt(n_tab / n_tfidf),
# щоб обидва блоки мали однаковий "голос" у регуляризованій моделі.
import scipy.sparse as sp
_n_tfidf = X_train_tfidf.shape[1]
_n_tab   = X_train_tab.shape[1]
_tfidf_scale = float(np.sqrt(_n_tab / _n_tfidf))  # ≈ 0.144
print(f'TF-IDF scale factor: {_tfidf_scale:.4f}  '
      f'(n_tfidf={_n_tfidf}, n_tab={_n_tab})')
X_train_tfidf_scaled = X_train_tfidf.multiply(_tfidf_scale)
X_test_tfidf_scaled  = X_test_tfidf.multiply(_tfidf_scale)
X_train_fused = hstack([X_train_tfidf_scaled, X_train_tab])
X_test_fused  = hstack([X_test_tfidf_scaled, X_test_tab])

# 5.  ВИПРАВЛЕННЯ 3: Злита щільна матриця (SVD тексту + таблична)
# У попередній версії X_train_fused_dense визначався, але не використовувався!
# Тепер він підставляється у нову модель HistGB (Злита щільна SVD).
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_train_text_dense = svd.fit_transform(X_train_tfidf)
X_test_text_dense  = svd.transform(X_test_tfidf)

X_train_fused_dense = np.hstack([X_train_tab_dense, X_train_text_dense])
X_test_fused_dense  = np.hstack([X_test_tab_dense, X_test_text_dense])

print(f'Таблична (розріджена): навчальна {X_train_tab.shape}')
print(f'Таблична (щільна)    : навчальна {X_train_tab_dense.shape}')
print(f'Злита (розріджена)   : навчальна {X_train_fused.shape}')
print(f'Злита (щільна SVD)   : навчальна {X_train_fused_dense.shape}')
print(f'\nSVD: пояснює {svd.explained_variance_ratio_.sum()*100:.1f}% дисперсії TF-IDF')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Година дня vs розподіл результатів
ax = axes[0]
hour_result = train_df.groupby(['hour', 'result']).size().unstack(fill_value=0)
hour_result_pct = hour_result.div(hour_result.sum(axis=1), axis=0)
hour_result_pct.plot(kind='bar', stacked=True, ax=ax, legend=False, colormap='tab10')
ax.set_title('Розподіл результатів за годиною дня')
ax.set_xlabel('Година')
ax.set_ylabel('Частка')
ax.tick_params(axis='x', rotation=45)

# 2. День тижня vs результати
ax = axes[1]
day_names = ['Пн','Вт','Ср','Чт','Пт','Сб','Нд']
dow_result = train_df.groupby(['dayofweek', 'result']).size().unstack(fill_value=0)
dow_result_pct = dow_result.div(dow_result.sum(axis=1), axis=0)
dow_result_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_xticks(range(7))
ax.set_xticklabels(day_names, rotation=0)
ax.set_title('Розподіл результатів за днем тижня')
ax.set_ylabel('Частка')
ax.legend(title='result', fontsize=7, loc='lower right')

plt.suptitle('Аналіз сигналу ознак — часові патерни', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Визначення моделей

Шість класифікаторів із чітко розмежованими ролями для прогнозування `result`:

### Текстові моделі
- **Naive Bayes**: базова лінія, бінаризований TF-IDF.  `alpha=1.0` (виправлено з 0.001)
- **Logistic Regression**: лінійна межа у просторі TF-IDF, `class_weight='balanced'`

### Табличні моделі
- **Random Forest**: ансамбль дерев, `class_weight='balanced'`
- **HistGradientBoosting**:  `max_iter=200, max_depth=4` (виправлено з `10, 1`)

### Змішані моделі
- **LR (Злита розріджена)**: TF-IDF + таблична матриця
- **HistGB (Злита щільна SVD)**:  SVD(100) + таблична — **нова модель**, виправляє баг невикористаної матриці

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import recall_score, make_scorer
import numpy as np
import math

def worst_class_recall(y_true, y_pred):
    return np.min(recall_score(y_true, y_pred, average=None, zero_division=0))

scorer = make_scorer(worst_class_recall)

#  ВИПРАВЛЕННЯ 7: sample_weight для HistGB (не підтримує class_weight)
# Клас 0 (Вирішено позитивно): 72.1%, клас 1 (Дано роз'яснення): 27.9%
# Без зважування HistGB завжди передбачає більшість → погана confusion matrix
from sklearn.utils.class_weight import compute_sample_weight
sample_weight_train = compute_sample_weight('balanced', y_train)

# ── Єдиний реєстр моделей ──────────────────────────────────────────────────
registry = [
    {
        'name': 'Naive Bayes (тільки текст)',
        'model': MultinomialNB(),
        'grid': {'alpha': [0.1, 0.5, 1.0, 5.0]},
        'X_train': X_train_tfidf.multiply(X_train_tfidf > 0),
        'X_test':  X_test_tfidf.multiply(X_test_tfidf > 0),
    },
    {
        'name': 'Logistic Regression (тільки текст)',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced',
                                    solver='saga', penalty='elasticnet',
                                    random_state=RANDOM_STATE),
        'grid': {'C': [0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.15, 0.5, 0.8]},
        'X_train': X_train_tfidf,
        'X_test':  X_test_tfidf,
    },
    {
        'name': 'Random Forest (тільки таблична)',
        'model': RandomForestClassifier(class_weight='balanced',
                                        random_state=RANDOM_STATE, n_jobs=-1),
        'grid': {'n_estimators': [100, 200], 'max_depth': [6, 12, None],
                 'min_samples_leaf': [10, 20]},
        'X_train': X_train_tab,
        'X_test':  X_test_tab,
    },
    {
        'name': 'HistGB (тільки таблична)',
        'model': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        'grid': {'max_iter': [100, 200], 'max_depth': [4, 10],
                 'learning_rate': [0.01, 0.05, 0.1],
                 'l2_regularization': [0.1, 1.0, 10.0]},
        'X_train': X_train_tab_dense,
        'X_test':  X_test_tab_dense,
        'sample_weight': sample_weight_train,  # Виправлення 7: збалансоване зважування
    },
    {
        'name': 'Logistic Regression (Злита: Текст + Таблична)',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced',
                                    solver='saga', penalty='elasticnet',
                                    random_state=RANDOM_STATE),
        'grid': {'C': [0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.15, 0.5, 0.8]},
        'X_train': X_train_fused,
        'X_test':  X_test_fused,
    },
    {
        'name': 'HistGB (Злита щільна: SVD + Таблична)',
        'model': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        'grid': {'max_iter': [100, 200], 'max_depth': [4, 10],
                 'learning_rate': [0.01, 0.05, 0.1],
                 'l2_regularization': [0.1, 1.0, 10.0]},
        'X_train': X_train_fused_dense,
        'X_test':  X_test_fused_dense,
        'sample_weight': sample_weight_train,  # Виправлення 7: збалансоване зважування
    },
]

def _grid_size(grid):
    return math.prod(len(v) for v in grid.values())

# ── Єдиний цикл налаштування ───────────────────────────────────────────────
print("Запуск оптимізації RandomizedSearch...\n")
models = {}   # побудовано тут, замінює обидва старих словники

for cfg in registry:
    name = cfg['name']
    print(f" {name} ...", end=' ', flush=True)

    search = RandomizedSearchCV(
        estimator=cfg['model'],
        param_distributions=cfg['grid'],
        n_iter=min(20, _grid_size(cfg['grid'])),  # дивись допоміжну функцію нижче
        cv=5,
        scoring={'worst_recall': scorer},
        refit='worst_recall',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    _sw = cfg.get('sample_weight', None)
    _fit_kw = {'sample_weight': _sw} if _sw is not None else {}
    search.fit(cfg['X_train'], y_train, **_fit_kw)

    models[name] = {
        'model':   search.best_estimator_,
        'X_train': cfg['X_train'],
        'X_test':  cfg['X_test'],
        'note':    f"Найкращі параметри: {search.best_params_}",
    }
    print(f"оцінка={search.best_score_:.4f}  {search.best_params_}")

---
## 8. Навчання та оцінювання

Кожна модель навчається і оцінюється за сімома метриками.  
**Ключова метрика** — Macro F1, яка рівномірно штрафує за погані результати на будь-якому класі.

**Базова лінія MAE**: для прогнозування `result` MAE не застосовується (немає порядку між класами).  
Замість неї використовуємо **Cohen's κ** як основну метрику збалансованості.

In [ ]:
results = {}
predictions = {}
probas = {}

y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))

def worst_class_recall_eval(y_true, y_pred):
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    return np.min(recalls)

for name, cfg in models.items():
    short = name.replace('\n', ' ')
    print(f'Навчання {short}...', end=' ', flush=True)

    model = cfg['model']
    Xtr   = cfg['X_train']
    Xte   = cfg['X_test']

    sw = cfg.get('sample_weight', None)
    if sw is not None:
        model.fit(Xtr, y_train, sample_weight=sw)
    else:
        model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    predictions[name] = y_pred

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(Xte)
    else:
        y_proba = np.zeros((len(y_test), len(classes)))
        y_proba[np.arange(len(y_pred)), y_pred] = 1.0

    if hasattr(model, 'classes_'):
        y_proba_aligned = np.zeros((y_proba.shape[0], len(classes)))
        for i, cls_idx in enumerate(model.classes_):
            y_proba_aligned[:, cls_idx] = y_proba[:, i]
    else:
        y_proba_aligned = y_proba

    probas[name] = y_proba_aligned

    try:
        roc_auc = roc_auc_score(
            y_test, y_proba_aligned,
            multi_class='ovr', average='macro'
        )
    except Exception:
        roc_auc = np.nan

    results[name] = {
        'Accuracy':        accuracy_score(y_test, y_pred),
        'Macro F1':        f1_score(y_test, y_pred, average='macro'),
        'Weighted F1':     f1_score(y_test, y_pred, average='weighted'),
        'Macro Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Macro Recall':    recall_score(y_test, y_pred, average='macro', zero_division=0),
        'Worst Recall':    worst_class_recall_eval(y_test, y_pred),  # <-- added
        'MCC':             matthews_corrcoef(y_test, y_pred),
        "Cohen's κ":       cohen_kappa_score(y_test, y_pred),
    }
    print(f"готово. Acc={results[name]['Accuracy']:.3f}  "
          f"MacroF1={results[name]['Macro F1']:.3f}")

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
print('\n=== Результати на тестовій вибірці (Березень 2026) ===')

higher_is_better = list(results_df.columns)
cols = ["Accuracy", "Macro F1", "Worst Recall"]  # лише ці колонки

display(
    results_df.style
        .highlight_max(subset=cols, axis=0, color="green")
        .highlight_min(subset=cols, axis=0, color="red")
        .format("{:.4f}")
)

---
## 9. Візуальне порівняння моделей

Три діаграми дають різні кути зору на результати:

1. **Гістограми метрик**: пряме порівняння Accuracy, Macro F1, MCC по моделях
2. **Cohen's κ**: ключова метрика збалансованості з базовими лініями
3. **Радарна діаграма**: профілі моделей по всіх метриках одночасно

In [ ]:
metrics_to_plot = ['Accuracy', 'Worst Recall']
model_labels = [n.replace('\n', '\n') for n in results_df.index]
palette = ['#e74c3c', '#e67e22', '#2980b9', '#27ae60', '#8e44ad', '#16a085']

# Змінено з (1, N) → (N, 1)
fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(10, 10))

# Переконуємось, що axes є ітерабельним при одному підграфіку
if len(metrics_to_plot) == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics_to_plot):
    vals = results_df[metric].values.astype(float)
    bars = ax.bar(range(len(vals)), vals,
                  color=palette[:len(vals)], alpha=0.88, edgecolor='white')

    ymin = max(0, float(np.nanmin(vals)) - 0.08)
    ymax = min(1.0, float(np.nanmax(vals)) + 0.08)
    ax.set_ylim(ymin, ymax)

    ax.set_title(metric, fontsize=10)
    ax.set_xticks(range(len(model_labels)))
    ax.set_xticklabels([m.split('\n')[0] for m in model_labels],
                       fontsize=8, rotation=30, ha='right')

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

fig.suptitle(
    'Порівняння моделей — Прогнозування результату розгляду\n'
    'Червоний/Помаранчевий = тільки текст  |  Зелений/Синій = тільки таблиця  |  Фіолетовий/Бірюзовий = злиті',
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
kappas = results_df["Cohen's κ"].values.astype(float)
bars = ax.bar(range(len(kappas)), kappas, color=palette[:len(kappas)], alpha=0.88, edgecolor='white')
ax.axhline(0, color='grey', linestyle='--', alpha=0.5, label='Базова лінія (κ=0, випадкова модель)')
ax.axhline(0.2, color='orange', linestyle=':', alpha=0.5, label='κ=0.2 (слабка згода)')
ax.set_ylim(-0.1, 1.05)
ax.set_title("Cohen's κ — Збалансована метрика узгодженості (вище = краще)\n"
             'κ=0: випадкова модель; κ=1: ідеальна модель')
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels([m.replace('\n', ' ') for m in model_labels],
                   fontsize=8, rotation=20, ha='right')
ax.set_ylabel("Cohen's κ")
ax.legend(fontsize=8)
for bar, v in zip(bars, kappas):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01, f'{v:.3f}',
            ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
radar_metrics = ['Accuracy', 'Macro F1', 'Weighted F1',
                 'Macro Precision', 'Macro Recall']
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + \
         [np.linspace(0, 2 * np.pi, N, endpoint=False)[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for (name, row), color in zip(results_df[radar_metrics].iterrows(), palette):
    values = row.values.tolist() + [row.values[0]]
    ax.plot(angles, values, 'o-', linewidth=1.8, color=color,
            label=name.replace('\n', ' '))
    ax.fill(angles, values, alpha=0.07, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Профілі моделей — Радарна діаграма\nЗовні = краще.',
             fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), fontsize=8)
plt.tight_layout()
plt.show()